# **Projet 3 - Prédiction de la consommation énergétique des bâtiments de Seattle**

 * Prédire la consommation énergétique des bâtiments non résidentiels de Seattle à partir de leurs caractéristiques structurelles.
 * Réaliser une analyse exploratoire afin d'identifier les principales tendances et anomalies des données.
 * Comparer plusieurs modèles de régression supervisée pour sélectionner le plus performant.
 * Identifier les variables ayant le plus d'influence sur la consommation énergétique.*
 

## **Objectf** 
Prédire la consommation énergétique (ou les émissions) des bâtiments non résidentiels afin d'aider la ville de Seattle à atteindre la neutralité carbone en 2050.

## **ÉTAPE 3** 

### **Prépararez les Features pour la modélisation**

#### **Importation des Modules**

In [2]:
#importation des librairies

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Importation des librairies OK")

Importation des librairies OK


In [3]:
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor


print("Importation des modules OK")

Importation des modules OK


#### **Chargement du dataset**

In [4]:
df = pd.read_csv("../data/processed/buildings_features.csv")

print("Importation du dataset terminée.")
print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")

display(df.head())

Importation du dataset terminée.
Dimensions du dataset : 3348 lignes, 30 colonnes


,datayear,buildingtype,primarypropertytype,councildistrictcode,neighborhood,latitude,longitude,yearbuilt,numberofbuildings,numberoffloors,...,energystarscore,siteenergyuse(kbtu),defaultdata,ghgemissionsintensity,usage_type,BuildingAge,ParkingRatio,BuildingRatio,IsMultiUse,NumberPropertyUses
0,2016,NonResidential,Hotel,7,DOWNTOWN,47.61220,-122.33799,1927,1.0,12,...,60.0,7226362.5,False,2.83,Mono-usage,89,0.000000,1.000000,0,1
1,2016,NonResidential,Hotel,7,DOWNTOWN,47.61317,-122.33393,1996,1.0,11,...,61.0,8387933.0,False,2.86,Multi-usages,20,0.145453,0.854547,1,3
2,2016,NonResidential,Hotel,7,DOWNTOWN,47.61393,-122.33810,1969,1.0,41,...,43.0,72587024.0,False,2.19,Mono-usage,47,0.205748,0.794252,0,1
3,2016,NonResidential,Hotel,7,DOWNTOWN,47.61412,-122.33664,1926,1.0,10,...,56.0,6794584.0,False,4.67,Mono-usage,90,0.000000,1.000000,0,1
4,2016,NonResidential,Hotel,7,DOWNTOWN,47.61375,-122.34047,1980,1.0,18,...,75.0,14172606.0,False,2.88,Multi-usages,36,0.353115,0.646885,1,3


### **Préparation des Features pour la modélisation**

#### **Séparation des variables x / y**

> Le jeu de données est séparé en :
>  - **X** : les variables explicatives utilisées pour entraîner le modèle ;
>  - **y** : la variable cible (**SiteEnergyUse(kBtu)**).

In [5]:
# Séparation des variables explicatives (X) et de la cible (y)

X = df.drop(columns=["siteenergyuse(kbtu)"])

y = df["siteenergyuse(kbtu)"]

print(f"Dimensions de X : {X.shape}")
print(f"Dimensions de y : {y.shape}")

Dimensions de X : (3348, 29)
Dimensions de y : (3348,)


### **Identification des variables catégorielles**

>  - Les variables catégorielles sont encodées à l'aide de **OneHotEncoder** afin de les rendre exploitables par les modèles de Machine Learning.
>  - Cette méthode évite d'introduire un ordre artificiel entre les différentes catégories.

In [6]:
# Identification des variables catégorielles

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Nombre de variables catégorielles : {len(categorical_features)}")

categorical_features

Nombre de variables catégorielles : 8


C:\Users\FR103217\AppData\Local\Temp\ipykernel_25076\1155983229.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


['buildingtype',
 'primarypropertytype',
 'neighborhood',
 'listofallpropertyusetypes',
 'largestpropertyusetype',
 'secondlargestpropertyusetype',
 'thirdlargestpropertyusetype',
 'usage_type']

In [7]:
# Identification des variables numériques

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print(f"Variables numériques : {len(numeric_features)}")
print(f"Variables catégorielles : {len(categorical_features)}")

Variables numériques : 20
Variables catégorielles : 8


### **Configuration du préprocessement des variables catégorielles**

>  - Les variables catégorielles sont transformées en variables numériques à l'aide de **OneHotEncoder**.
>  - Cette étape permet aux algorithmes de Machine Learning de traiter correctement les différentes catégories sans introduire de relation d'ordre entre elles.

In [8]:
# Préprocessement des variables catégorielles

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. 

## **Conclusion**

>  - Les variables explicatives (X) et la variable cible (y) ont été préparées pour l'entraînement.
>  - Les variables catégorielles ont été identifiées afin d'être encodées avec OneHotEncoder.
>  - Le préprocesseur est désormais prêt pour être intégré au pipeline de modélisation.

> Le jeu de données est désormais prêt pour la phase de modélisation.

In [9]:
print(df.shape)
display(df.head())

(3348, 30)


,datayear,buildingtype,primarypropertytype,councildistrictcode,neighborhood,latitude,longitude,yearbuilt,numberofbuildings,numberoffloors,...,energystarscore,siteenergyuse(kbtu),defaultdata,ghgemissionsintensity,usage_type,BuildingAge,ParkingRatio,BuildingRatio,IsMultiUse,NumberPropertyUses
0,2016,NonResidential,Hotel,7,DOWNTOWN,47.61220,-122.33799,1927,1.0,12,...,60.0,7226362.5,False,2.83,Mono-usage,89,0.000000,1.000000,0,1
1,2016,NonResidential,Hotel,7,DOWNTOWN,47.61317,-122.33393,1996,1.0,11,...,61.0,8387933.0,False,2.86,Multi-usages,20,0.145453,0.854547,1,3
2,2016,NonResidential,Hotel,7,DOWNTOWN,47.61393,-122.33810,1969,1.0,41,...,43.0,72587024.0,False,2.19,Mono-usage,47,0.205748,0.794252,0,1
3,2016,NonResidential,Hotel,7,DOWNTOWN,47.61412,-122.33664,1926,1.0,10,...,56.0,6794584.0,False,4.67,Mono-usage,90,0.000000,1.000000,0,1
4,2016,NonResidential,Hotel,7,DOWNTOWN,47.61375,-122.34047,1980,1.0,18,...,75.0,14172606.0,False,2.88,Multi-usages,36,0.353115,0.646885,1,3


## **Sauvegarde du dataset**

In [10]:
from pathlib import Path

# Création du dossier
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Sauvegarde du jeu de données préparé
df.to_csv(
    "../data/processed/buildings_model.csv",
    index=False
)

print("Jeu de données préparé sauvegardé avec succès.")

Jeu de données préparé sauvegardé avec succès.
